# "THE POUR IS RIGHT" - Wine Pricer Capstone

# Week 7: fine-tune an open-source model

This week we fine-tune an open-source model on our wines, to see whether a small specialist can beat the frontier models. The score to beat is Claude Sonnet 5 at \$9.06.

# Order of play

DAY 1: QLoRA (the idea)  
DAY 2: Prompt data and Base Model  
DAY 3: Train Part 1  
DAY 4: Train Part 2  
DAY 5: Eval  

## DAY 2: the prompt data

This notebook follows `week7/day2.ipynb` and runs on the laptop for free. It turns each wine into the exact text the model will train on, and pushes that to the Hugging Face Hub so the Colab notebooks can load it.

## The base model

Ed used `meta-llama/Llama-3.2-3B`. That model is gated: you must request access from Meta. We use **`Qwen/Qwen3-4B-Base`** instead: 4 billion parameters, text only, Apache-2.0 licence, no approval needed.

It is the **Base** version, not the chat version. A base model simply continues whatever text it is given, which is what we want: we will give it a wine description ending in "Price is \$" and train it to continue with the number.

In [ ]:
import os
from dotenv import load_dotenv
from huggingface_hub import HfApi
from pricer.items import Item
from tqdm.notebook import tqdm
from transformers import AutoTokenizer
import matplotlib.pyplot as plt

load_dotenv(override=True)

In [ ]:
username = HfApi().whoami()["name"]
dataset = f"{username}/wines_lite"

train, val, test = Item.from_hub(dataset)
items = train + val + test

print(f"Loaded {len(train):,} training items, {len(val):,} validation items, {len(test):,} test items")

## The tokenizer

A model does not read characters or words. It reads tokens: pieces of text, each with a number. Every model family has its own tokenizer, so we must measure our text with the one our base model uses.

Only the tokenizer is downloaded here (a few MB), not the model.

In [ ]:
BASE_MODEL = "Qwen/Qwen3-4B-Base"

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)

### How does it write a price?

This matters more than it looks. Ed chose Llama partly because it has one token for every number up to 999, so a price is a single token to predict.

Qwen writes numbers one digit at a time. The next cell shows `36.00` becoming five tokens. So our model has to get several tokens right in a row to produce a price, and when we generate we must allow up to six tokens (Ed allowed five in total).

In [ ]:
for text in ["36.00", "8.00", "125.00", "Price is $36.00"]:
    token_ids = tokenizer.encode(text, add_special_tokens=False)
    print(f"{text!r} is {len(token_ids)} tokens: {[tokenizer.decode([token_id]) for token_id in token_ids]}")

## How long are our summaries?

Training time and memory both grow with the number of tokens, so we want a cap that keeps nearly every wine whole.

In [ ]:
token_counts = [item.count_tokens(tokenizer) for item in tqdm(items)]

In [ ]:
plt.figure(figsize=(15, 6))
plt.title(f"Tokens in Summary: Avg {sum(token_counts)/len(token_counts):,.1f} and highest {max(token_counts):,}\n")
plt.xlabel('Number of tokens in summary')
plt.ylabel('Count')
plt.hist(token_counts, rwidth=0.7, color="skyblue", bins=range(0, 220, 10))
plt.show()

Ed's cutoff was 110 tokens. Our summaries are a little longer, averaging about 102 tokens, so we use 150. That trims only 22 wines (0.1%), and what gets trimmed is the end of the Verdict line.

In [ ]:
CUTOFF = 150
cut = len([count for count in token_counts if count > CUTOFF])
print(f"With this CUTOFF, we will truncate {cut:,} items which is {cut/len(items):.1%}")

## Make the prompts

Each wine becomes two pieces of text:

- the **prompt**: a question, the summary, and the words "Price is \$"
- the **completion**: the price

For training and validation the price is rounded to whole dollars and written like `36.00`. That gives the model an easier, more regular target. For the test set the completion is the exact price, because that is what we score against.

The `.00` looks wasteful, but it does a job: it marks where the number ends, so the model learns to stop.

In [ ]:
print(train[0].summary)

In [ ]:
for item in tqdm(train+val):
    item.make_prompts(tokenizer, CUTOFF, True)
for item in tqdm(test):
    item.make_prompts(tokenizer, CUTOFF, False)

In [ ]:
print("PROMPT:")
print(test[7].prompt)
print("COMPLETION:")
print(test[0].completion)

In [ ]:
print("PROMPT:")
print(train[7].prompt)
print("COMPLETION:")
print(train[0].completion)

In [ ]:
prompt_token_counts = [item.count_prompt_tokens(tokenizer) for item in tqdm(items)]

In [ ]:
plt.figure(figsize=(15, 6))
plt.title(f"Tokens: Avg {sum(prompt_token_counts)/len(prompt_token_counts):,.1f} and highest {max(prompt_token_counts):,}\n")
plt.xlabel('Number of tokens in prompt and the completion')
plt.ylabel('Count')
plt.hist(prompt_token_counts, rwidth=0.7, color="gold", bins=range(0, 220, 10))
plt.show()

The highest value in the chart above is the longest text the model will ever see in training. We will set the model's maximum length just above it in the training notebook.

## Push to the Hub

This dataset has only two columns, `prompt` and `completion`, which is the format the training library expects.

In [ ]:
dataset = f"{username}/wines_prompts_lite"

Item.push_prompts_to_hub(dataset, train, val, test)
print(f"Pushed to https://huggingface.co/datasets/{dataset}")